## Experiments #3 and #5: random control

Does it matter *which* sentences get the extra training, or is any extra training enough?
Per seed and embedding mode we (1) train the LSTM on the full training set, then (2) give it an **extra fine-tuning phase on a random subset** of the training set, as large as the hard subset used by the targeted runs, and (3) score on the validation set (overall / hard / non-hard, macro F1).

- **#3**: FinBERT embeddings, frozen
- **#5**: FinBERT embeddings, fine-tuned

`EXTRA_EPOCHS` and `EXTRA_LR` must be identical to the targeted runs (#2 / #4). Test set is not used.

In [2]:
import os
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from tqdm import tqdm

from utils.hard_samples_identify import get_hard_sample_indices
from utils.random_control import (
    build_finbert_embedding_matrix, embedding_name, make_dummy_embedding_matrix, oof_logreg_probs,
    prepare_data, run_random_control, summarize_seeds, to_metrics_rows, update_metrics_csv,
)

SEEDS = [42, 43, 44, 45, 46]
EXTRA_EPOCHS = 3   # TODO confirm: must match targeted runs #2 / #4
EXTRA_LR = 1e-4    # TODO confirm: must match targeted runs #2 / #4
USE_DUMMY_EMBEDDINGS = True  # True = quick pipeline test with random vectors; results are NOT saved

### Data (shared preprocessing, tokenizer fitted on train only)

In [3]:
train = pd.read_csv("../data/train.csv")
val = pd.read_csv("../data/val.csv")
data = prepare_data(train, val)
print(data["X_train"].shape, data["X_val"].shape, data["vocab_size"])

(2901, 64) (967, 64) 7081


### Hard subsets
- **Validation** hard / non-hard split: bottom 20% margin of the main baseline (TF-IDF + LogReg) validation probabilities, so every experiment is scored on the same sentences.
- **Training** hard set: out-of-fold predictions (5-fold). Only its *size* is used here: the random control gets the same number of sentences.

In [4]:
val_probs = pd.read_csv("../results/probs/baseline_logreg_val.csv")
assert (val_probs["sentence"].values == val["sentence"].values).all()
val_hard_idx, _ = get_hard_sample_indices(val_probs[["p_negative", "p_neutral", "p_positive"]].to_numpy())

oof = oof_logreg_probs(train)
hard_train_idx, _ = get_hard_sample_indices(oof)
print(f"val hard: {len(val_hard_idx)} / {len(val)}   |   train hard (= random subset size): {len(hard_train_idx)} / {len(train)}")

val hard: 194 / 967   |   train hard (= random subset size): 580 / 2901


### FinBERT embeddings mapped onto our vocabulary

In [5]:
if USE_DUMMY_EMBEDDINGS:
    embedding_matrix = make_dummy_embedding_matrix(data["vocab_size"])
else:
    embedding_matrix, stats = build_finbert_embedding_matrix(
        data["tok"].word_index, cache_path="../models/finbert_embedding_matrix.npz")
    print(stats)  # how many words are a single FinBERT word piece vs split into several
print(embedding_matrix.shape)

(7081, 768)


### Run: #3 (frozen) and #5 (fine-tuned), all seeds

In [6]:
rows = []
for trainable in [False, True]:   # False = #3, True = #5
    for seed in tqdm(SEEDS, desc=embedding_name(trainable)):
        rows += run_random_control(data, embedding_matrix, trainable, seed,
                                   hard_train_idx, val_hard_idx, EXTRA_EPOCHS, EXTRA_LR)
seed_df = pd.DataFrame(rows)
seed_df

finbert_frozen:   0%|          | 0/5 [00:00<?, ?it/s]

finbert_finetuned: 100%|██████████| 5/5 [02:43<00:00, 32.79s/it]


,seed,embedding_config,n_extra,extra_epochs,extra_lr,overlap_with_hard,embedding_change,phase,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
0,42,finbert_frozen,580,3,0.0001,0.215517,0.000000,stage1,0.719752,0.614197,0.515464,0.397586,0.771022,0.674902,0.464286,0.811572,0.566735
1,42,finbert_frozen,580,3,0.0001,0.215517,0.000000,after_extra,0.718718,0.644856,0.505155,0.406057,0.772316,0.708033,0.543478,0.801972,0.589118
2,43,finbert_frozen,580,3,0.0001,0.194828,0.000000,stage1,0.703206,0.619451,0.469072,0.401109,0.761966,0.684498,0.487310,0.795341,0.575701
3,43,finbert_frozen,580,3,0.0001,0.194828,0.000000,after_extra,0.708376,0.613125,0.469072,0.379320,0.768435,0.682988,0.477157,0.804173,0.558045
4,44,finbert_frozen,580,3,0.0001,0.172414,0.000000,stage1,0.713547,0.639671,0.489691,0.418855,0.769728,0.703137,0.538813,0.806020,0.574181
5,44,finbert_frozen,580,3,0.0001,0.172414,0.000000,after_extra,0.721820,0.659650,0.510309,0.454049,0.774903,0.716621,0.580000,0.803361,0.595588
6,45,finbert_frozen,580,3,0.0001,0.224138,0.000000,stage1,0.698035,0.607277,0.448454,0.371633,0.760673,0.675687,0.500000,0.796163,0.525667
7,45,finbert_frozen,580,3,0.0001,0.224138,0.000000,after_extra,0.707342,0.625542,0.463918,0.386907,0.768435,0.693722,0.508108,0.793806,0.574713
8,46,finbert_frozen,580,3,0.0001,0.196552,0.000000,stage1,0.730093,0.651682,0.500000,0.419002,0.787840,0.720098,0.547170,0.814516,0.593361
9,46,finbert_frozen,580,3,0.0001,0.196552,0.000000,after_extra,0.720786,0.649929,0.500000,0.441276,0.776197,0.710359,0.551402,0.804636,0.593750


### Results (mean and standard deviation over seeds, after the extra phase)

In [7]:
mean, std = summarize_seeds(seed_df)
display(mean.round(3))
display(std.round(3))

# Effect of the extra phase itself: stage 1 (before) vs after_extra
seed_df.groupby(["embedding_config", "phase"])[["f1_overall", "f1_hard", "f1_nonhard"]].mean().round(3)

,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
embedding_config,,,,,,,,,
finbert_finetuned,0.747,0.685,0.518,0.468,0.804,0.749,0.593,0.822,0.641
finbert_frozen,0.715,0.639,0.490,0.414,0.772,0.702,0.532,0.802,0.582


,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
embedding_config,,,,,,,,,
finbert_finetuned,0.004,0.003,0.013,0.018,0.005,0.007,0.015,0.004,0.012
finbert_frozen,0.007,0.019,0.022,0.033,0.004,0.014,0.040,0.004,0.016


f1_overall  f1_hard  f1_nonhard
embedding_config  phase                                       
finbert_finetuned after_extra       0.685    0.468       0.749
                  stage1            0.667    0.450       0.730
finbert_frozen    after_extra       0.639    0.414       0.702
                  stage1            0.626    0.402       0.692

### Save (per-seed details + one averaged row per experiment in the shared metrics.csv)

In [8]:
if USE_DUMMY_EMBEDDINGS:
    print("Dummy embeddings: nothing saved.")
else:
    seed_df.to_csv("../results/random_control_seeds.csv", index=False)
    update_metrics_csv("../results/metrics.csv", to_metrics_rows(mean))

Dummy embeddings: nothing saved.
